In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
.appName('Spark Caching')\
.getOrCreate()

26/06/09 03:36:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
spark

In [5]:
!hadoop fs -ls /user/tanuj/1MB



Found 5 items
-rw-r--r--   2 tanujkumawat3008 hadoop    1058478 2026-06-03 15:34 /user/tanuj/1MB/customers.csv
-rw-r--r--   2 tanujkumawat3008 hadoop     837537 2026-06-03 15:34 /user/tanuj/1MB/items.csv
-rw-r--r--   2 tanujkumawat3008 hadoop     864432 2026-06-03 15:34 /user/tanuj/1MB/orders.csv
-rw-r--r--   2 tanujkumawat3008 hadoop     855780 2026-06-03 15:34 /user/tanuj/1MB/payments.csv
-rw-r--r--   2 tanujkumawat3008 hadoop     772763 2026-06-03 15:34 /user/tanuj/1MB/shippings.csv


## RDD CACHING 

### for small file 

In [12]:
customers_rdd = spark.sparkContext.textFile('/user/tanuj/1MB/customers.csv')

In [13]:
customers_filtered = customers_rdd.filter(lambda row: 'Mumbai' in row)
mapped = customers_filtered.map(lambda row:(row.split(',')[0],1))
reduced = mapped.reduceByKey(lambda x,y : x+y)

In [14]:
reduced.count() #1st run -> 7.5 seconds

2093

In [15]:
reduced.count() #2nd run -> 1.8 seconds

2093

In [16]:
reduced.count() #3rd run -> 325 mili seconds

2093

In [17]:
reduced.count() #4th run ->287 mili seconds

2093

In [18]:
reduced.cache()

PythonRDD[14] at RDD at PythonRDD.scala:53

In [20]:
reduced.count()

2093

In [21]:
reduced.unpersist() # to uncache

PythonRDD[14] at RDD at PythonRDD.scala:53

In [22]:
reduced.count()

2093

### for large file

In [24]:
!hadoop fs -ls /data


Found 6 items
-rw-r--r--   2 root hadoop    1058478 2026-06-04 17:20 /data/customers.csv
-rw-r--r--   2 root hadoop   26214400 2026-06-04 17:48 /data/customers_500.csv
-rw-r--r--   2 root hadoop        209 2026-06-07 07:42 /data/dates_data.csv
-rw-r--r--   2 root hadoop     653137 2026-06-06 16:16 /data/messy_customer_data.csv
drwxr-xr-x   - root hadoop          0 2026-06-06 16:55 /data/write_output.csv
drwxr-xr-x   - root hadoop          0 2026-06-06 17:00 /data/write_output1.csv


In [25]:
large_rdd = spark.sparkContext.textFile('/data/customers_500.csv')

In [26]:
customers_filtered = large_rdd.filter(lambda row: 'Mumbai' in row)
mapped = customers_filtered.map(lambda row:(row.split(',')[0],1))
reduced = mapped.reduceByKey(lambda x,y : x+y)

In [27]:
reduced.count()

51051

In [28]:
reduced.cache()

PythonRDD[24] at RDD at PythonRDD.scala:53

In [29]:
reduced.count()

51051

In [30]:
reduced.count()

51051

In [31]:
reduced.unpersist()

PythonRDD[24] at RDD at PythonRDD.scala:53

## DATAFRAME CACHING

In [32]:
df = spark.read.format('csv').option('header','true').option('inferSchema','true').load('/user/tanuj/1MB/customers.csv')

In [34]:
df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-05-17|     true|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-11-29|     true|
|          2|Customer_2|Hyderabad|Maharashtra|  India|       2023-02-08|     true|
|          3|Customer_3|Bangalore|Maharashtra|  India|       2023-03-28|    false|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-06-04|    false|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [35]:
df.cache()

DataFrame[customer_id: int, name: string, city: string, state: string, country: string, registration_date: date, is_active: boolean]

In [36]:
tail_df = df.orderBy('customer_id',ascending=False)

In [38]:
tail_df.show(5)

+-----------+--------------+---------+-----------+-------+-----------------+---------+
|customer_id|          name|     city|      state|country|registration_date|is_active|
+-----------+--------------+---------+-----------+-------+-----------------+---------+
|      17331|Customer_17331|Bangalore|West Bengal|  India|       2023-08-20|    false|
|      17330|Customer_17330|  Chennai|  Telangana|  India|       2023-12-19|     true|
|      17329|Customer_17329|  Chennai|      Delhi|  India|       2023-11-06|     true|
|      17328|Customer_17328|     Pune|  Telangana|  India|       2023-07-04|     true|
|      17327|Customer_17327|Hyderabad|  Telangana|  India|       2023-02-11|     true|
+-----------+--------------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [39]:
df.count()

17332

In [40]:
df.count()

17332

In [41]:
df.unpersist()

DataFrame[customer_id: int, name: string, city: string, state: string, country: string, registration_date: date, is_active: boolean]

In [ ]:
spark.stop()